In [74]:
import boto3
from dotenv import load_dotenv
import os
import sqlalchemy
import pymysql

load_dotenv()

aws_access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')
region_name = os.getenv('region_name')
master_user_password = os.getenv('master_db_password')
master_username = os.getenv('master_db_name')
port = os.getenv('port') 

# Créez une session boto3
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name
)

# Créez un client RDS
rds_client = session.client('rds')

# Paramètres de la base de données
db_instance_identifier = 'dbkayak'
db_instance_class = 'db.t4g.micro'  # Free Tier instance type
engine = 'mysql'
allocated_storage = 20  # Free Tier allows up to 20 GB

In [90]:
# Vérifiez si l'instance RDS existe déjà
try:
    db_instance_info = rds_client.describe_db_instances(DBInstanceIdentifier=db_instance_identifier)
    print(f"L'instance RDS '{db_instance_identifier}' existe déjà.")
except rds_client.exceptions.DBInstanceNotFoundFault:
    # Créez la base de données RDS si elle n'existe pas
    try:
        response = rds_client.create_db_instance(
            DBInstanceIdentifier=db_instance_identifier,
            DBInstanceClass=db_instance_class,
            Engine=engine,
            MasterUsername=master_username,
            MasterUserPassword=master_user_password,
            AllocatedStorage=allocated_storage,
            BackupRetentionPeriod=7,  # Number of days to retain backups
            MultiAZ=False,  # Free Tier does not support Multi-AZ deployments
            PubliclyAccessible=True,  # Set to False if you don't want the DB to be publicly accessible
            StorageType='gp2',  # General Purpose SSD
        )
        print("Creating RDS instance...")
        print(response)
    except Exception as e:
        print(f"Error creating RDS instance: {e}")

L'instance RDS 'dbkayak' existe déjà.


In [75]:
# Obtenez les informations de l'instance RDS
try:
    db_instance_info = rds_client.describe_db_instances(DBInstanceIdentifier=db_instance_identifier)
    endpoint = db_instance_info['DBInstances'][0]['Endpoint']['Address']
    port = db_instance_info['DBInstances'][0]['Endpoint']['Port']
    name=db_instance_info['DBInstances'][0]['DBInstanceIdentifier']
    print(f"Name: {name}")  
    print(f"Endpoint: {endpoint}")
    print(f"Port: {port}")
except Exception as e:
    print(f"Error retrieving DB instance info: {e}")

Name: dbkayak
Endpoint: dbkayak.c3g8yqkisjyz.eu-west-3.rds.amazonaws.com
Port: 3306


In [91]:
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError

new_database_name = 'dbkayak'

# Chaîne de connexion sans base de données pour la création de la base
base_connection_string = f"mysql+pymysql://{master_username}:{master_user_password}@{endpoint}:{port}"
database_connection_string = f"mysql+pymysql://{master_username}:{master_user_password}@{endpoint}:{port}/{new_database_name}"

# Créer le moteur SQLAlchemy sans base de données spécifique
base_engine = create_engine(base_connection_string)

def check_and_create_database(engine, db_name):
    """Vérifie si la base de données existe et la crée si elle n'existe pas."""
    database_exists_query = text(f"SELECT SCHEMA_NAME FROM INFORMATION_SCHEMA.SCHEMATA WHERE SCHEMA_NAME = :db_name")

    try:
        with engine.connect() as connection:
            # Vérifie si la base de données existe
            result = connection.execute(database_exists_query, {"db_name": db_name}).fetchone()
            if result:
                print(f"La base de données '{db_name}' existe déjà.")
            else:
                # Crée la base de données si elle n'existe pas
                connection.execute(text(f"CREATE DATABASE {db_name}"))
                print(f"Base de données '{db_name}' créée avec succès.")
    except OperationalError as e:
        print(f"Erreur lors de la vérification ou de la création de la base de données : {e}")

def connect_to_database(connection_url):
    """Essaie de se connecter à une base de données et gère les erreurs de connexion."""
    engine = create_engine(connection_url)
    try:
        with engine.connect() as connection:
            print("Connexion réussie à la base de données MySQL!")
    except OperationalError as e:
        print(f"Erreur lors de la connexion à la base de données : {e}")

# Vérifiez et créez la base de données si nécessaire
check_and_create_database(base_engine, new_database_name)

# Créer le moteur pour se connecter à la nouvelle base de données
connect_to_database(database_connection_string)


La base de données 'dbkayak' existe déjà.
Connexion réussie à la base de données MySQL!


In [82]:
import pandas as pd
import requests

In [83]:
url='https://tmopenlabbucket.s3.eu-west-3.amazonaws.com/City_Meteo_Rank_Booking.csv'

df = pd.read_csv(url,index_col=0)

In [84]:
df.head(10)

,City,City_latitude,City_longitude,City_CCM,Hotels_name,Hotels_url,Hotels_score,Hotels_description,Hotels_latitude,Hotels_longitude
0,Aigues Mortes,43.566152,4.191540,0.915,"['La Chambre Côté Piscine', 'La Maison du Môle...",['https://www.booking.com/hotel/fr/la-chambre-...,"[10.0, 9.6, 9.0, 9.1, 8.3, 9.2, 9.7, 9.7, 8.5,...",['Boasting air-conditioned accommodation with ...,"[43.5720862, 43.5518696, 43.5659428, 43.565764...","[4.1963892, 4.16810244, 4.1924, 4.188757, 4.17..."
1,Aix en Provence,43.529842,5.447474,0.745,"['Hôtel Le Mozart', 'Hôtel de France', 'Radiss...",['https://www.booking.com/hotel/fr/le-mozart.e...,"[8.4, 7.2, 8.0, 8.0, 9.2, 9.0, 8.4, 8.0, 8.4, ...","['Located 500 metres from the Granet Museum, H...","[43.52191762, 43.52693137, 43.528614, 43.54750...","[5.45787156, 5.44626698, 5.426005, 5.3873295, ..."
2,Amiens,49.894171,2.295695,0.829,"['Hôtel Spa Marotte', ""L'Hortillon"", '4 Ch - 1...",['https://www.booking.com/hotel/fr/ha-tel-maro...,"[8.0, 9.5, 8.4, 8.1, 9.0, 9.8, 8.5, 8.1, 8.5, ...","['Located in the centre of Amiens, 500 metres ...","[49.89105067, 49.89562864, 49.9071282, 49.8897...","[2.30115093, 2.31126979, 2.3172668, 2.30249941..."
3,Annecy,45.899235,6.128885,0.757,"['ibis Annecy Centre Vieille Ville', 'Magic Mo...",['https://www.booking.com/hotel/fr/ibis-annecy...,"[8.3, 8.8, 8.9, 7.9, 8.3, 8.4, 8.3, 9.1, 8.5, ...","['Located just 700 metres from Lake Annecy, ib...","[45.8982388, 45.90013154, nan, 45.90303773, 45...","[6.12250954, 6.11441195, nan, 6.12265706, 6.12..."
4,Avignon,43.949249,4.805901,0.609,"['Hôtel Le Bristol', 'Au cœur des Papes, dans ...",['https://www.booking.com/hotel/fr/hotel-brist...,"[8.3, 8.1, 9.3, 9.7, 8.4, 8.4, 9.3, 7.9, 9.6, ...","[None, 'Au cœur des Papes, dans bâtisse de cha...","[43.94403213, 43.9461706, 43.947213, 43.945187...","[4.80533168, 4.8101887, 4.8120952, 4.7996263, ..."
5,Bayeux,49.276462,-0.702474,0.900,['Historic 18th-Century Mansion - 3 BR - Bayeu...,['https://www.booking.com/hotel/fr/bayeux-norm...,"[10.0, 9.8, 9.4, 8.1, 9.8, 9.5, 9.6, 8.8, 8.5,...","['Set in Bayeux, in a historic building, 300 m...","[49.2776109, 49.2756423, 49.2758646, nan, 49.2...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
6,Bayonne,43.494514,-1.473666,0.918,"['Hostel20 Bayonne', 'Duplex Typique Centre Ba...",['https://www.booking.com/hotel/fr/hostel-20-b...,"[7.7, 8.3, 9.4, 8.3, 7.3, 8.0, 8.2, 8.0, 7.6, ...",['Located 15 km from Biarritz La Négresse Trai...,"[43.49535, 43.489894, 43.4944671, 43.49349109,...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
7,Biarritz,43.471144,-1.552727,0.915,"['Hôtel Jules Verne Biarritz', 'Le Garage Biar...",['https://www.booking.com/hotel/fr/karitza-bia...,"[8.3, 9.2, 7.6, 8.5, 8.4, 9.1, 9.2, 9.3, 9.1, ...","[None, None, 'Located on the seafront, Hôtel F...","[43.47853954, 43.490592, 43.48222961, 43.48534...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
8,Bormes les Mimosas,43.150697,6.341928,0.874,['Appartement climatisé Bord de Mer La Faviere...,['https://www.booking.com/hotel/fr/apartement-...,"[8.7, 9.0, 8.5, 7.4, 8.4, 9.1, 9.0, 9.3, 8.6, ...","['Offering a garden and garden view, Apparteme...","[nan, 43.12333323, 43.15127099, 43.15187759, 4...","[nan, 6.35251241, 6.34190677, 6.34313598, 6.33..."
9,Carcassonne,43.213036,2.349107,0.712,"['Le Cœur de la Bastide - Adult only', 'Les Fl...",['https://www.booking.com/hotel/fr/le-coeur-de...,"[9.4, 9.1, 8.0, 8.2, 9.5, 9.7, 9.7, 8.5, 8.1, ...",['Le Cœur de la Bastide - Adult only in Carcas...,"[43.214181, 43.20934977, 43.20927546, nan, 43....","[2.35006, 2.36376107, 2.36338303, nan, 2.35690..."


In [94]:
engine = create_engine(database_connection_string)

In [95]:
# Insérez le DataFrame dans la base de données MySQL
try:
    df.to_sql(name='dbkayak', con=engine, if_exists='replace', index=False)
    print("DataFrame inséré avec succès dans la table 'dbkayak' de la base de données MySQL.")
except Exception as e:
    print(f"Erreur lors de l'insertion du DataFrame dans la base de données: {e}")

DataFrame inséré avec succès dans la table 'dbkayak' de la base de données MySQL.


In [96]:
from sqlalchemy import text

stmt = text("SELECT * FROM dbkayak.dbkayak LIMIT 5")

df = pd.read_sql_query(con=engine.connect(), sql=stmt)

df

,City,City_latitude,City_longitude,City_CCM,Hotels_name,Hotels_url,Hotels_score,Hotels_description,Hotels_latitude,Hotels_longitude
0,Aigues Mortes,43.566152,4.191540,0.915,"['La Chambre Côté Piscine', 'La Maison du Môle...",['https://www.booking.com/hotel/fr/la-chambre-...,"[10.0, 9.6, 9.0, 9.1, 8.3, 9.2, 9.7, 9.7, 8.5,...",['Boasting air-conditioned accommodation with ...,"[43.5720862, 43.5518696, 43.5659428, 43.565764...","[4.1963892, 4.16810244, 4.1924, 4.188757, 4.17..."
1,Aix en Provence,43.529842,5.447474,0.745,"['Hôtel Le Mozart', 'Hôtel de France', 'Radiss...",['https://www.booking.com/hotel/fr/le-mozart.e...,"[8.4, 7.2, 8.0, 8.0, 9.2, 9.0, 8.4, 8.0, 8.4, ...","['Located 500 metres from the Granet Museum, H...","[43.52191762, 43.52693137, 43.528614, 43.54750...","[5.45787156, 5.44626698, 5.426005, 5.3873295, ..."
2,Amiens,49.894171,2.295695,0.829,"['Hôtel Spa Marotte', ""L'Hortillon"", '4 Ch - 1...",['https://www.booking.com/hotel/fr/ha-tel-maro...,"[8.0, 9.5, 8.4, 8.1, 9.0, 9.8, 8.5, 8.1, 8.5, ...","['Located in the centre of Amiens, 500 metres ...","[49.89105067, 49.89562864, 49.9071282, 49.8897...","[2.30115093, 2.31126979, 2.3172668, 2.30249941..."
3,Annecy,45.899235,6.128885,0.757,"['ibis Annecy Centre Vieille Ville', 'Magic Mo...",['https://www.booking.com/hotel/fr/ibis-annecy...,"[8.3, 8.8, 8.9, 7.9, 8.3, 8.4, 8.3, 9.1, 8.5, ...","['Located just 700 metres from Lake Annecy, ib...","[45.8982388, 45.90013154, nan, 45.90303773, 45...","[6.12250954, 6.11441195, nan, 6.12265706, 6.12..."
4,Avignon,43.949249,4.805901,0.609,"['Hôtel Le Bristol', 'Au cœur des Papes, dans ...",['https://www.booking.com/hotel/fr/hotel-brist...,"[8.3, 8.1, 9.3, 9.7, 8.4, 8.4, 9.3, 7.9, 9.6, ...","[None, 'Au cœur des Papes, dans bâtisse de cha...","[43.94403213, 43.9461706, 43.947213, 43.945187...","[4.80533168, 4.8101887, 4.8120952, 4.7996263, ..."
